### Train

In [19]:
import numpy as np
import torch

lr=1e-3 


class EarlyStoppingF1:
    def __init__(self, patience=20, verbose=False, path='best_model__0.pt'):
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_f1_max = 0
        self.path = path

    def __call__(self, val_f1, model):
        score = val_f1
        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_f1, model)
        elif score < self.best_score:
            self.counter += 1
            if self.verbose:
                print(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_f1, model)
            self.counter = 0

    def save_checkpoint(self, val_f1, model):
        if self.verbose:
            print(f'Validation F1 increased ({self.val_f1_max:.6f} --> {val_f1:.6f}). Saving model...')
        torch.save(model.state_dict(), self.path)
        self.val_f1_max = val_f1
        
        
        

from sklearn.metrics import f1_score, accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns


device = DEV

model = ImprovedTemporalClassifier()
model.to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=lr)


# ============================================================
# ============================================================
epochs=400
# ============================================================
# ============================================================

stopper = EarlyStoppingF1(patience=20, verbose=True)
    
for epoch in range(epochs):
        # --- Training Phase ---
        model.train()
        train_loss = 0
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # --- Validation Phase ---
        model.eval()
        all_preds = []
        all_labels = []
        val_loss = 0
        
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_loss += criterion(logits, y).item()
                preds = torch.argmax(logits, dim=1)
                
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y.cpu().numpy())

        # Calculate Metrics
        v_f1 = f1_score(all_labels, all_preds, average='macro')
        v_acc = accuracy_score(all_labels, all_preds)
        
        print(f"Epoch {epoch+1}: Train Loss: {train_loss/len(train_loader):.4f} | "
              f"Val Loss: {val_loss/len(val_loader):.4f} | Val F1: {v_f1:.4f} | Val Acc: {v_acc:.4f}")

        # Check Early Stopping
        stopper(v_f1, model)
        if stopper.early_stop:
            print("Early stopping triggered")
            break

# ============================================================
# ============================================================
# ============================================================
# Load the best weights before final evaluation
model.load_state_dict(torch.load('best_model__0.pt'))        
# ============================================================
# ============================================================
# ============================================================


Epoch 1: Train Loss: 0.8868 | Val Loss: 0.6607 | Val F1: 0.3585 | Val Acc: 0.7367
Validation F1 increased (0.000000 --> 0.358452). Saving model...
Epoch 2: Train Loss: 0.6226 | Val Loss: 0.6040 | Val F1: 0.4770 | Val Acc: 0.7583
Validation F1 increased (0.358452 --> 0.477015). Saving model...
Epoch 3: Train Loss: 0.5817 | Val Loss: 0.6136 | Val F1: 0.5142 | Val Acc: 0.7432
Validation F1 increased (0.477015 --> 0.514195). Saving model...
Epoch 4: Train Loss: 0.5646 | Val Loss: 0.5769 | Val F1: 0.5718 | Val Acc: 0.7692
Validation F1 increased (0.514195 --> 0.571762). Saving model...
Epoch 5: Train Loss: 0.5472 | Val Loss: 0.6067 | Val F1: 0.5952 | Val Acc: 0.7543
Validation F1 increased (0.571762 --> 0.595153). Saving model...
Epoch 6: Train Loss: 0.5300 | Val Loss: 0.5834 | Val F1: 0.6074 | Val Acc: 0.7695
Validation F1 increased (0.595153 --> 0.607446). Saving model...
Epoch 7: Train Loss: 0.5292 | Val Loss: 0.5868 | Val F1: 0.5907 | Val Acc: 0.7711
EarlyStopping counter: 1 out of 20
E

<All keys matched successfully>